# Fusion Design Lab — GRPO Training

Train an LLM to optimize stellarator fusion reactor designs using **GRPO** (Group Relative Policy Optimization) with **HF TRL**.

The agent interacts with a constrained optimization environment where it adjusts 4 geometric knobs of a stellarator boundary, aiming to **minimize max elongation** while satisfying 3 hard physics constraints:
- `aspect_ratio ≤ 4.0`
- `average_triangularity ≤ -0.5`
- `abs(edge_iota_over_nfp) ≥ 0.3`

Each episode has **6 evaluations** budgeted. The agent produces a plan of actions and the environment scores it via the `constellaration` physics verifier.

**Environment deployed at**: https://creativeengineer-fusion-design-lab.hf.space

**Runtime**: Select GPU (T4 or better) via `Runtime > Change runtime type`.

## 1. Install Dependencies

In [ ]:
%%capture

# Build deps for constellaration (booz-xform compiles from source)
!apt-get update -qq && apt-get install -y -qq cmake ninja-build g++ gfortran libnetcdf-dev libnetcdff-dev > /dev/null

!{sys.executable} -m pip install trl peft bitsandbytes datasets matplotlib accelerate
!{sys.executable} -m pip install "transformers==5.2.0" "huggingface-hub>=1.3.0,<2.0"

## 2. Load Model with LoRA

In [ ]:
import importlib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen3.5-4B"
MAX_SEQ_LENGTH = 2048

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

attn_impl = "flash_attention_2" if importlib.util.find_spec("flash_attn") else "sdpa"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation=attn_impl,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()
print(f"Model loaded: {MODEL_NAME} (attn: {attn_impl})")

## 3. Setup Stellarator Environment

Install the environment package directly from the HF Space repository so training runs locally (no network latency per step). The package also includes the typed `FusionLabClient` and Pydantic models for remote OpenEnv sessions.

In [ ]:
%%capture
# Install the fusion-design-lab environment (includes constellaration physics engine)
# This takes ~3 minutes due to booz-xform compilation
!{sys.executable} -m pip install "fusion-design-lab @ git+https://huggingface.co/spaces/CreativeEngineer/fusion-design-lab"

In [ ]:
import json
from typing import Final

from fusion_lab.llm_agent import (
    RUN_DIRECTIONS,
    RUN_MAGNITUDES,
    RUN_PARAMETERS,
    build_prompt,
    parse_action_plan,
    run_episode_with_actions,
)
from fusion_lab.models import StellaratorAction
from server.contract import RESET_SEEDS
from server.environment import BUDGET, StellaratorEnvironment

RUN_ACTION_SPECS: Final[list[dict[str, str]]] = [
    {"intent": "run", "parameter": p, "direction": d, "magnitude": m}
    for p in RUN_PARAMETERS
    for d in RUN_DIRECTIONS
    for m in RUN_MAGNITUDES
]

AVAILABLE_ACTIONS: Final[list[dict[str, str]]] = RUN_ACTION_SPECS + [
    {"intent": "restore_best"},
]
# Quick smoke test
env = StellaratorEnvironment()
obs = env.reset(seed=0)
print(
    f"Environment ready. Initial score: {obs.p1_score:.4f}, feasibility: {obs.p1_feasibility:.4f}"
)
print(f"Budget: {obs.budget_remaining}, Constraints satisfied: {obs.constraints_satisfied}")

## 4. Prompt Template & Action Parsing

Each training sample is a prompt describing the stellarator task and initial state. The model generates a plan of actions to optimize the design.

In [ ]:
# Shared helper smoke test
env = StellaratorEnvironment()
obs = env.reset(seed=0)
prompt = build_prompt(obs)
print(prompt[:500])
print("...")

sample_completion = json.dumps(
    [
        {
            "intent": "run",
            "parameter": "triangularity_scale",
            "direction": "increase",
            "magnitude": "small",
        },
        {"intent": "submit"},
    ]
)
print(parse_action_plan(sample_completion))

## 5. Training Dataset

Create prompts from all 3 reset seeds. Each prompt is an initial observation that the model must optimize.

In [ ]:
from datasets import Dataset

prompts = []
for seed_idx in range(len(RESET_SEEDS)):
    obs = StellaratorEnvironment().reset(seed=seed_idx)
    prompt = build_prompt(obs)
    # Repeat each seed to create a larger training set
    for _ in range(50):
        prompts.append({"prompt": prompt, "seed_idx": seed_idx})

dataset = Dataset.from_list(prompts)
dataset = dataset.shuffle(seed=42)
print(f"Training dataset: {len(dataset)} samples from {len(RESET_SEEDS)} seeds")

## 6. Reward Function

The environment reward executes each generated action plan in the stellarator environment and returns the cumulative low-fidelity Reward V2 from the live environment. The environment's built-in reward decomposes feasibility (+3/-3 crossing bonuses, official feasibility progress, new-best / near-feasible bonuses, weighted triangularity/aspect/iota repair terms), objective (max elongation improvement plus best-feasible-score bonus), step costs, and bounded anti-stagnation penalties — see `server/environment.py:_compute_reward_breakdown(...)`.

For the current training workflow, the notebook ignores `submit` and does not auto-submit. GRPO therefore optimizes the low-fidelity `run` path only. The live observation telemetry still exposes `reward_breakdown` and `action_monitor` for debugging reward behavior.


In [ ]:
import traceback


def environment_reward_fn(
    completions: list[str], seed_idx: list[int] | None = None, **kwargs
) -> list[float]:
    """Execute each action plan in the environment and return cumulative reward.

    This is the sole GRPO training signal in the notebook. It uses the live
    low-fidelity environment reward path and ignores submit so the trainer
    optimizes only the `run` surface. Empty or unparseable outputs still
    receive a trainer-side fallback penalty of -3.0.
    """
    rewards = []
    seeds = seed_idx if seed_idx is not None else [0] * len(completions)
    for i, completion in enumerate(completions):
        try:
            actions = parse_action_plan(completion)
            if len(actions) == 0:
                rewards.append(-3.0)
                continue
            trace = run_episode_with_actions(
                actions,
                seed_idx=int(seeds[i]) % len(RESET_SEEDS),
            )
            rewards.append(trace.total_reward)
        except Exception:
            traceback.print_exc()
            rewards.append(-3.0)
    return rewards


# Test reward function with a hand-crafted plan
test_plan = json.dumps(
    [
        {
            "intent": "run",
            "parameter": "triangularity_scale",
            "direction": "increase",
            "magnitude": "small",
        },
        {
            "intent": "run",
            "parameter": "rotational_transform",
            "direction": "increase",
            "magnitude": "medium",
        },
    ]
)
print(f"Environment reward (low-fi only): {environment_reward_fn([test_plan], seed_idx=[0])}")

# Test short plan with no explicit submit
test_short = json.dumps(
    [
        {
            "intent": "run",
            "parameter": "triangularity_scale",
            "direction": "increase",
            "magnitude": "medium",
        },
    ]
)
print(f"Environment reward (short plan): {environment_reward_fn([test_short], seed_idx=[0])}")

## 7. GRPO Training

Train the model using Group Relative Policy Optimization. GRPO generates multiple completions per prompt and updates the policy toward higher-reward completions.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

MAX_PROMPT_LENGTH = 768
MAX_COMPLETION_LENGTH = MAX_SEQ_LENGTH - MAX_PROMPT_LENGTH

training_args = GRPOConfig(
    output_dir="./grpo_fusion_output",
    learning_rate=5e-5,
    num_generations=8,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    max_steps=60,
    temperature=1.0,
    logging_steps=1,
    save_steps=20,
    bf16=True,
    report_to="none",
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[environment_reward_fn],
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO training...")
train_result = trainer.train()
print(f"Training complete. Total steps: {train_result.global_step}")

## 8. Training Results

Visualize reward improvement over training steps.

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [entry["step"] for entry in log_history if "loss" in entry]
losses = [entry["loss"] for entry in log_history if "loss" in entry]

# Extract reward metrics if available
reward_steps = [
    entry["step"]
    for entry in log_history
    if "reward" in entry or "rewards/environment_reward_fn" in entry
]
rewards = [
    entry.get("reward", entry.get("rewards/environment_reward_fn", 0))
    for entry in log_history
    if "reward" in entry or "rewards/environment_reward_fn" in entry
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(steps, losses, "b-", alpha=0.7)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("GRPO Training Loss")
axes[0].grid(True, alpha=0.3)

if rewards:
    axes[1].plot(reward_steps, rewards, "g-o", alpha=0.7, markersize=3)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Mean Reward")
    axes[1].set_title("Environment Reward Over Training")
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Reward metrics not logged", ha="center", va="center")

plt.suptitle("Fusion Design Lab — GRPO Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")

## 9. Evaluate Trained Policy

Generate action plans from the trained model and compare against random baselines.

In [ ]:
import random

model.eval()


def reward_term_summary(step_or_obs: object) -> str:
    breakdown_obj = getattr(step_or_obs, "reward_breakdown")
    breakdown = (
        breakdown_obj.model_dump() if hasattr(breakdown_obj, "model_dump") else breakdown_obj
    )
    terms = []
    for key, value in breakdown.items():
        if key in {
            "intent",
            "total",
            "evaluation_failed",
            "recovered_from_failure",
            "reference_constraints_satisfied",
            "reference_score",
            "reference_feasibility",
            "reference_max_elongation",
            "initial_reference_score",
            "terminal_score_ratio",
        }:
            continue
        if isinstance(value, (int, float)) and float(value) != 0.0:
            terms.append(f"{key}={float(value):+.3f}")
    return ", ".join(terms) if terms else "none"


def run_episode_with_model(seed_idx: int) -> tuple[float, list[str]]:
    """Run one episode using the trained model."""
    env = StellaratorEnvironment()
    obs = env.reset(seed=seed_idx)
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION_LENGTH,
            temperature=0.7,
            do_sample=True,
        )
    completion = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    actions = parse_action_plan(completion)
    episode = run_episode_with_actions(actions, seed_idx=seed_idx)
    trace = [
        (
            f"{step.action_label} → reward={step.reward:.3f} "
            f"score={step.p1_score:.4f} feasible={step.constraints_satisfied} "
            f"terms={reward_term_summary(step)}"
        )
        for step in episode.steps
    ]
    return episode.total_reward, trace


def run_random_episode(seed_idx: int) -> float:
    """Run one episode with random actions for comparison."""
    actions = [StellaratorAction(**random.choice(RUN_ACTION_SPECS)) for _ in range(BUDGET)]
    return run_episode_with_actions(actions, seed_idx=seed_idx).total_reward


# Evaluate
print("=" * 60)
print("TRAINED MODEL EPISODES")
print("=" * 60)
trained_rewards = []
for seed in range(len(RESET_SEEDS)):
    reward, trace = run_episode_with_model(seed)
    trained_rewards.append(reward)
    print(f"\nSeed {seed} — Total reward: {reward:.3f}")
    for line in trace:
        print(f"  {line}")

print(f"\nMean trained reward: {sum(trained_rewards) / len(trained_rewards):.3f}")

print("\n" + "=" * 60)
print("RANDOM BASELINE (10 episodes per seed)")
print("=" * 60)
random_rewards = []
for seed in range(len(RESET_SEEDS)):
    seed_rewards = [run_random_episode(seed) for _ in range(10)]
    random_rewards.extend(seed_rewards)
    print(
        f"Seed {seed} — Mean: {sum(seed_rewards) / len(seed_rewards):.3f}, Best: {max(seed_rewards):.3f}"
    )

print(f"\nMean random reward: {sum(random_rewards) / len(random_rewards):.3f}")
print(f"Mean trained reward: {sum(trained_rewards) / len(trained_rewards):.3f}")

## 10. Connect to Deployed HF Space

Demonstrate connecting to the live environment on Hugging Face Spaces through the typed OpenEnv client and running the trained model against it.

In [ ]:
import requests

from fusion_lab.client import FusionLabClient

HF_SPACE_URL = "https://creativeengineer-fusion-design-lab.hf.space"

# Check health
health = requests.get(f"{HF_SPACE_URL}/health").json()
print(f"HF Space status: {health['status']}")

# Get task description
task = requests.get(f"{HF_SPACE_URL}/task").json()
print(f"\nTask: {task['description']}")
print(f"Constraints: {task['constraints']}")
print(f"Budget: {task['budget']}")

with FusionLabClient(base_url=HF_SPACE_URL) as env:
    reset_result = env.reset(seed=42)
    remote_obs = reset_result.observation
    print(f"\nRemote reset — max_elongation: {remote_obs.max_elongation:.4f}")
    print(f"  aspect_ratio: {remote_obs.aspect_ratio:.4f}")
    print(f"  constraints_satisfied: {remote_obs.constraints_satisfied}")
    print(f"  budget_remaining: {remote_obs.budget_remaining}")

    # Generate an action plan from the trained model
    prompt = build_prompt(remote_obs)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_COMPLETION_LENGTH, temperature=0.7, do_sample=True
        )
    completion = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    actions = parse_action_plan(completion)

    print(f"\nTrained model generated {len(actions)} actions for remote env:")
    for i, action in enumerate(actions[:BUDGET], start=1):
        if action.intent == "submit":
            continue
        result = env.step(action)
        step_obs = result.observation
        reward = float(result.reward) if result.reward is not None else 0.0
        print(
            f"  Step {i}: {action.intent} {action.parameter or ''} "
            f"{action.direction or ''} {action.magnitude or ''} "
            f"→ reward={reward:.3f}, score={step_obs.p1_score:.4f}, terms={reward_term_summary(step_obs)}"
        )
        if result.done:
            print(f"  Episode done. Final score: {step_obs.p1_score:.4f}")
            break
print("\nEnvironment is live and accessible for training and evaluation.")